# 8교시. 실무 적용 시나리오 설계 및 최종 정리

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/master/colab/08_business_application.ipynb)

## 오늘 꼭 할 일

견적서·신청서·거래명세서 실물 사진을 비교하고 첫 PoC 한 가지를 고릅니다.

1. 제공 예제로 결과를 먼저 만듭니다.
2. 화면에서 이번 교시의 핵심 결과 한 가지를 확인합니다.
3. 시간이 남으면 다른 공개·비식별 자료로 반복하고 차이를 기록합니다.

**끝났다는 증거:** 화면의 `✅ 실습 완료`와
`course_outputs/poc_candidate_card.md` 파일

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

필수 실습에서는 제공 예제를 사용합니다. 다른 자료를 사용한 경우에는
화면에 표시된 파일명이 내가 선택한 파일과 같은지 먼저 확인합니다.
모델 설치가 3분 이상 진행되지 않으면 실행을 중지하고 제공 예제로
핵심 단계를 계속합니다.


## 이 노트북에서 내가 하는 일

- **필수 실습:** 견적서·신청서·거래명세서 중 한 문서를 골라 작은 PoC 후보 카드를 만듭니다.
- **내가 바꾸는 곳:** 문서 종류·점수·검토자·중단 조건만 채웁니다.
- **인터넷 자료로 다시 실험:** 공개 PDF나 비식별 캡처 한 장을 찾아 필요한 필드와 실패 조건을 카드에 추가합니다.

먼저 제공 예제로 끝까지 실행해 `✅ 실습 완료`를 확인하세요. 그다음
[공개·비식별 실습 자료 찾기](https://github.com/leecks1119/document_ai_lecture/blob/master/docs/public_practice_sources.md)를 보고
입력 한 장만 바꾸어 다시 실행합니다. 2교시에서 만든 결과 파일은
3~7교시에 이어 쓸 수 있습니다. 매 교시 마지막의 **다른 자료 실험
기록**에서 잘된 점과 실패한 점을 남깁니다.

> `🟢 그대로 실행하는 셀`은 수정하지 않습니다. `🟠 내가 짧게 바꾸는
> 셀`만 필수이고, `🔵 원하면 바꾸는 셀`은 시간이 남을 때 합니다.
> 정답은 모두 공개되어 있으므로 정답을 먼저 복사하고 결과를 관찰해도 됩니다.

## 코드 셀을 읽는 방법

각 코드 셀의 맨 위에는 `코드 읽기` 주석이 있습니다.

1. `수정하지 않습니다`라고 적힌 셀은 설명을 읽고 그대로 실행합니다.
2. 주황색 필수 `TODO`만 채웁니다. 파란색 선택 `TODO`는 건너뛰어도 됩니다.
3. 실행 출력에서 `코드 읽는 법`과 `확인할 결과`를 다시 확인합니다.
4. `단계 실행 완료`가 나온 뒤 다음 코드 셀로 이동합니다.
5. 길고 어려운 준비 코드는 접혀 있습니다. 제목 왼쪽의 화살표를 눌러
   펼칠 수 있지만, 처음에는 펼치지 않아도 됩니다.

Python 문법 전체를 먼저 이해할 필요는 없습니다. 변수에 어떤 값이 들어가고,
실행 뒤 어떤 결과가 달라지는지를 중심으로 읽습니다.


In [ ]:
#@title 🟢 0. 실습 환경 준비 — 그대로 실행 { display-mode: "form" }
def _show_learning_message(markdown_text):
    try:
        from IPython.display import Markdown, display
        display(Markdown(markdown_text))
    except ImportError:
        print(markdown_text)


def show_lab_step(
    current,
    total,
    title,
    action,
    expected,
    code_help,
    edit_kind,
):
    cell_kind = {
        "required": "🟠 내가 짧게 바꾸는 셀",
        "optional": "🔵 원하면 바꾸는 셀",
        "none": "🟢 그대로 실행하는 셀",
    }[edit_kind]
    _show_learning_message(
        f"""---
### {cell_kind} · {current}/{total} · {title}

**지금 할 일:** {action}

**코드 읽는 법:** {code_help}

**이 단계에서 확인할 결과:** {expected}
"""
    )


def complete_lab_step(current, total, expected):
    next_action = (
        "결과를 확인한 뒤 다음 코드 셀을 실행하세요."
        if current < total
        else "마지막 실습 완료 문구와 산출물 파일을 확인하세요."
    )
    _show_learning_message(
        f"""> ✅ **{current}/{total} 단계 실행 완료**
>
> **결과 확인:** {expected}
>
> **다음 행동:** {next_action}
"""
    )

# ── 코드 읽기 ─────────────────────────────────────────────
# 업무 문서 샘플과 PoC 결과를 저장할 공통 폴더·자료 로더를 준비합니다. 설정 코드이므로 수정하지 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(1, 6, '공통 환경 준비', '업무 문서 샘플과 PoC 카드를 위한 공통 환경을 준비합니다.', 'Python·Platform·공통 작업 폴더가 표시되어야 합니다.', '업무 문서 샘플과 PoC 결과를 저장할 공통 폴더·자료 로더를 준비합니다. 설정 코드이므로 수정하지 않습니다.', 'none')

import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
AUTOMATED_CHECK = os.getenv("COURSE_VALIDATE_EXAMPLE") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or AUTOMATED_CHECK:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 자료 선택에서 "
            "'제공 예제'를 고르거나 파일을 다시 선택하세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if AUTOMATED_CHECK:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())

COURSE_ASSET_BASE_URL = (
    "https://raw.githubusercontent.com/leecks1119/"
    "document_ai_lecture/master/"
)

def load_course_assets(*relative_paths):
    if AUTOMATED_CHECK:
        local_root = os.getenv("COURSE_LOCAL_ASSET_ROOT")
        if not local_root:
            raise RuntimeError(
                "자동 검증용 COURSE_LOCAL_ASSET_ROOT가 필요합니다."
            )
        root = Path(local_root)
        return {
            path: (root / path).read_bytes()
            for path in relative_paths
        }

    import requests

    loaded = {}
    missing = []
    for path in relative_paths:
        try:
            response = requests.get(
                COURSE_ASSET_BASE_URL + path,
                timeout=30,
            )
            response.raise_for_status()
            loaded[path] = response.content
        except requests.RequestException as exc:
            print(f"자동 다운로드 실패: {Path(path).name} · {exc}")
            missing.append(path)

    if missing:
        from google.colab import files

        expected = ", ".join(Path(path).name for path in missing)
        print("다음 파일을 저장소에서 내려받아 선택하세요:", expected)
        uploaded = files.upload()
        uploaded_by_name = {
            Path(name).name: content
            for name, content in uploaded.items()
        }
        for path in missing:
            filename = Path(path).name
            if filename not in uploaded_by_name:
                raise FileNotFoundError(
                    f"{filename}이 선택되지 않았습니다."
                )
            loaded[path] = uploaded_by_name[filename]

    return loaded

complete_lab_step(1, 6, 'Python·Platform·공통 작업 폴더가 표시되어야 합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `EXTENSION_IMAGE_PATHS`가 문서별 사진 경로를 연결합니다. 반복문은 세 이미지를 같은 크기로 줄여 비교하기 쉽게 표시합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(2, 6, '실물형 문서 사진 비교', '견적서·신청서·거래명세서 사진을 차례로 관찰합니다.', '세 문서 이름·이미지 크기·미리보기가 보여야 합니다.', '`EXTENSION_IMAGE_PATHS`가 문서별 사진 경로를 연결합니다. 반복문은 세 이미지를 같은 크기로 줄여 비교하기 쉽게 표시합니다.', 'none')

import io
from PIL import Image
try:
    from IPython.display import display
except ImportError:
    display = lambda image: None

EXTENSION_IMAGE_PATHS = {
    "quotation": "sample_docs/extensions/quotation_photo.png",
    "application": "sample_docs/extensions/application_form_photo.png",
    "transaction_statement": (
        "sample_docs/extensions/transaction_statement_photo.png"
    ),
}
EXTENSION_EXAMPLES = {'quotation': {'name': '견적서', 'fields': ['문서번호', '공급자', '수신', '견적일', '품목', '총액'], 'rules': ['수량×단가=품목금액', '공급가액+부가세=총액'], 'risk': '총액 오류는 구매 의사결정에 직접 영향'}, 'application': {'name': '신청서', 'fields': ['신청번호', '신청자', '소속', '신청 과정', '승인'], 'rules': ['필수 동의', '관리자 승인 상태'], 'risk': '개인정보와 승인 누락을 사람이 확인'}, 'transaction_statement': {'name': '거래명세서', 'fields': ['문서번호', '공급자', '거래일', '품목', '세액', '총액'], 'rules': ['품목 합계=공급가액', '공급가액+세액=총액'], 'risk': '표 행·열 대응이 어긋나면 정산 오류'}}
extension_assets = load_course_assets(
    *EXTENSION_IMAGE_PATHS.values()
)
for key, path in EXTENSION_IMAGE_PATHS.items():
    image = Image.open(io.BytesIO(extension_assets[path])).convert("RGB")
    image.thumbnail((320, 400))
    print(key, image.size)
    display(image)

complete_lab_step(2, 6, '세 문서 이름·이미지 크기·미리보기가 보여야 합니다.')


## 형식이 바뀌면 생기는 어려움

- **Excel**: 수식, 병합 셀, 숨김 시트, 숫자 서식
- **Word**: 머리글, 텍스트박스, 변경 추적, 이미지로 삽입된 본문
- **PDF**: 텍스트·스캔 혼합 페이지, 암호, 깨진 문자맵
- **PPT**: 그룹 도형, 읽기 순서, 발표자 노트
- **표 캡처**: 셀 관계가 사라져 행·열 위상을 다시 복원해야 함


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `xml_text_count()`는 Office 내부 XML을 셉니다. 이어서 견적서·신청서·거래명세서의 검증 Python과 정답 JSON을 별도
# ZIP으로 묶습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(3, 6, 'Office 형식과 확장 예제 체험', 'Office 내부 구조를 검사하고 문서별 Python·JSON 예제를 묶습니다.', 'Office ZIP과 `business_document_code_examples.zip`을 확인합니다.', '`xml_text_count()`는 Office 내부 XML을 셉니다. 이어서 견적서·신청서·거래명세서의 검증 Python과 정답 JSON을 별도 ZIP으로 묶습니다.', 'none')

import re
import zipfile

OFFICE_PATHS = [
    "sample_docs/formats/quotation.xlsx",
    "sample_docs/formats/application_form.docx",
    "sample_docs/formats/transaction_statement.pdf",
    "sample_docs/formats/table_summary.pptx",
]
CODE_EXAMPLE_PATHS = [
    "src/document_examples.py",
    "sample_outputs/extensions/quotation.json",
    "sample_outputs/extensions/application.json",
    "sample_outputs/extensions/transaction_statement.json",
]
all_assets = load_course_assets(
    *OFFICE_PATHS,
    *CODE_EXAMPLE_PATHS,
)
office_assets = {
    path: all_assets[path]
    for path in OFFICE_PATHS
}
office_dir = OUTPUT_DIR / "office_format_samples"
office_dir.mkdir(exist_ok=True)
for path, payload in office_assets.items():
    (office_dir / Path(path).name).write_bytes(payload)

def xml_text_count(path, prefix, text_tag):
    with zipfile.ZipFile(path) as archive:
        names = [
            name for name in archive.namelist()
            if name.startswith(prefix) and name.endswith(".xml")
        ]
        text_count = 0
        for name in names:
            xml = archive.read(name).decode("utf-8", errors="ignore")
            text_count += len(re.findall(text_tag, xml))
        return len(names), text_count

xlsx_sheets, xlsx_values = xml_text_count(
    office_dir / "quotation.xlsx",
    "xl/worksheets/",
    r"<x:(?:v|f)>",
)
docx_parts, docx_text = xml_text_count(
    office_dir / "application_form.docx",
    "word/document",
    r"<w:t",
)
pptx_slides, pptx_text = xml_text_count(
    office_dir / "table_summary.pptx",
    "ppt/slides/slide",
    r"<a:t>",
)
pdf_bytes = (office_dir / "transaction_statement.pdf").read_bytes()
print("Excel:", xlsx_sheets, "개 시트 XML · 값/수식", xlsx_values)
print("Word:", docx_text, "개 본문 텍스트 run · 이미지 본문 여부 확인")
print("PDF:", pdf_bytes[:5], "· 텍스트층 샘플")
print("PPT:", pptx_slides, "개 슬라이드 · 텍스트", pptx_text)

office_bundle = OUTPUT_DIR / "office_format_samples.zip"
with zipfile.ZipFile(office_bundle, "w") as archive:
    for path in sorted(office_dir.iterdir()):
        archive.write(path, path.name)
print("실제 파일 4종 묶음:", office_bundle)
download_artifact(office_bundle)

code_example_dir = OUTPUT_DIR / "business_document_code_examples"
code_example_dir.mkdir(exist_ok=True)
for path in CODE_EXAMPLE_PATHS:
    target = code_example_dir / Path(path).name
    target.write_bytes(all_assets[path])
    if target.suffix == ".json":
        payload = json.loads(all_assets[path].decode("utf-8"))
        print(
            "확장 JSON:",
            payload["document_type"],
            "· 필드",
            len(payload),
        )

code_example_bundle = (
    OUTPUT_DIR / "business_document_code_examples.zip"
)
with zipfile.ZipFile(code_example_bundle, "w") as archive:
    for path in sorted(code_example_dir.iterdir()):
        archive.write(path, path.name)
print("문서별 Python·JSON 예제:", code_example_bundle)
download_artifact(code_example_bundle)

complete_lab_step(3, 6, 'Office ZIP과 `business_document_code_examples.zip`을 확인합니다.')


## 내가 직접 만드는 PoC 카드

`candidate`는 `quotation`, `application`, `transaction_statement`
중 하나입니다. 점수는 1~5점이며 오류 영향과 예외 빈도는 낮을수록
첫 PoC에 유리합니다.


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `candidate`, `score`, `review_owner`, `stop_condition`의 빈칸만 채웁니다. 모델 정확도보다 작은 PoC를
# 운영할 조건을 정하는 단계입니다.
# ──────────────────────────────────────────────────────────
show_lab_step(4, 6, '내 PoC 후보 입력', '대상 문서·점수·검토자·중단 조건을 직접 정합니다.', '빈칸 안내 또는 내가 입력한 PoC 조건이 표시되어야 합니다.', '`candidate`, `score`, `review_owner`, `stop_condition`의 빈칸만 채웁니다. 모델 정확도보다 작은 PoC를 운영할 조건을 정하는 단계입니다.', 'required')

# TODO: 내 업무 후보와 점수·검토자·중단 조건을 채우세요.
candidate = None
score = {
    "반복량": None,
    "필드 안정성": None,
    "오류 영향": None,
    "예외 빈도": None,
    "사람 검토 가능성": None,
}
review_owner = None
stop_condition = None
if candidate is None or any(value is None for value in score.values()):
    print("빈칸이 있습니다. 아래 힌트·전체 정답과 비교하세요.")

complete_lab_step(4, 6, '빈칸 안내 또는 내가 입력한 PoC 조건이 표시되어야 합니다.')


<details>
<summary>힌트와 전체 정답 보기</summary>

예시는 거래명세서를 한 장씩 처리하고 정산 담당자가 검토하는 작은
PoC입니다. 값이 맞지 않거나 원본 근거가 없으면 저장을 중단합니다.
</details>


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# 점수 조건이 맞으면 `작게 시작 가능`, 아니면 `조건 재검토`를 선택합니다. 결과와 검토·중단 조건을 Markdown PoC 카드로 저장합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(5, 6, 'PoC 후보 카드 완성', '점수 규칙으로 작은 PoC의 시작 여부를 판단해 저장합니다.', '후보 카드·`작게 시작 가능` 또는 `조건 재검토`·파일 경로를 확인합니다.', '점수 조건이 맞으면 `작게 시작 가능`, 아니면 `조건 재검토`를 선택합니다. 결과와 검토·중단 조건을 Markdown PoC 카드로 저장합니다.', 'none')

from textwrap import dedent

candidate = candidate or "transaction_statement"
if candidate not in EXTENSION_EXAMPLES:
    raise ValueError(
        "candidate는 quotation, application, "
        "transaction_statement 중 하나여야 합니다."
    )
default_score = {
    "반복량": 4,
    "필드 안정성": 4,
    "오류 영향": 2,
    "예외 빈도": 3,
    "사람 검토 가능성": 5,
}
score = {
    key: (
        int(value)
        if value is not None
        else default_score[key]
    )
    for key, value in score.items()
}
if not all(1 <= value <= 5 for value in score.values()):
    raise ValueError("모든 점수는 1~5 사이여야 합니다.")
review_owner = review_owner or "정산 담당자"
stop_condition = (
    stop_condition
    or "필수값·합계·원본 근거 중 하나라도 틀리면 자동 저장 중단"
)
example = EXTENSION_EXAMPLES[candidate]
recommendation = (
    "작게 시작 가능"
    if (
        score["반복량"] >= 4
        and score["필드 안정성"] >= 3
        and score["오류 영향"] <= 3
        and score["예외 빈도"] <= 3
        and score["사람 검토 가능성"] >= 4
    )
    else "조건 재검토"
)
card = f'''# 문서 자동화 PoC 후보 카드

| 항목 | 내용 |
| --- | --- |
| 선택 문서 | {example["name"]} |
| 추출 필드 | {", ".join(example["fields"])} |
| 검증 규칙 | {" / ".join(example["rules"])} |
| 틀렸을 때 영향 | {example["risk"]} |
| 입력 제한 | 승인된 비식별 한 장 |
| 최종 산출물 | 사람 승인 후 Excel |
| 사람 검토자 | {review_owner} |
| 중단 조건 | {stop_condition} |
| 점수 | {" / ".join(f"{key} {value}" for key, value in score.items())} |
| 제안 | {recommendation} |

## 첫 PoC 통과 기준

- 같은 양식 30장을 모아 정답표와 비교한다.
- 필드별 정확도뿐 아니라 수정률과 처리시간을 기록한다.
- 오류 시 자동 저장하지 않고 검토 대기열로 보낸다.
- 개인정보·보존·삭제 정책을 먼저 승인받는다.
'''
output_path = OUTPUT_DIR / "poc_candidate_card.md"
output_path.write_text(dedent(card), encoding="utf-8")
print(dedent(card))
print("✅ 실습 완료:", output_path)
download_artifact(output_path)

complete_lab_step(5, 6, '후보 카드·`작게 시작 가능` 또는 `조건 재검토`·파일 경로를 확인합니다.')


## 선택 실험: 다른 자료로 한 번 더 확인하기

필수 실습을 먼저 끝낸 뒤, 인터넷에서 찾은 공개 문서나 개인정보를
가린 자료 한 장으로 같은 과정을 반복합니다. 결과가 잘 나오지 않아도
실패한 위치와 다음 질문을 남기면 실험이 완료됩니다.


In [ ]:
#@title 🔵 선택: 다른 자료 실험 기록 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# 이 셀은 선택 실험 기록지입니다. 위쪽 입력칸만 채우면 자료 출처, 잘된 점, 실패한 점, 다음 질문을 Markdown 파일로 저장합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(6, 6, '다른 자료 실험 기록', '인터넷에서 찾은 공개 자료나 비식별 자료의 결과를 네 줄로 정리합니다.', '`lesson08_research_note.md` 파일과 기록 내용이 표시되어야 합니다.', '이 셀은 선택 실험 기록지입니다. 위쪽 입력칸만 채우면 자료 출처, 잘된 점, 실패한 점, 다음 질문을 Markdown 파일로 저장합니다.', 'optional')

# RESEARCH_NOTE_CELL
# TODO(선택): 다른 자료로 다시 실험했다면 아래 입력칸만 채우세요.
자료_구분 = "제공 예제" #@param ["제공 예제", "공개 웹 자료", "비식별 개인 자료", "회사 승인 자료"]
자료_이름_또는_URL = "" #@param {type:"string"}
문서_종류 = "영수증" #@param ["영수증", "견적서", "신청서", "거래명세서", "표 캡처", "기타"]
잘된_점 = "" #@param {type:"string"}
실패한_점 = "" #@param {type:"string"}
다음_질문 = "" #@param {type:"string"}

research_focus = '조사한 문서가 작은 PoC에 적합한 이유와 중단해야 할 조건을 기록합니다.'
note = f'''# {문서_종류} 실험 기록

- 자료 구분: {자료_구분}
- 자료 이름 또는 원문 URL: {자료_이름_또는_URL or "미입력"}
- 이번 교시 관찰 질문: {research_focus}
- 잘된 점: {잘된_점 or "미입력"}
- 실패하거나 이상한 점: {실패한_점 or "미입력"}
- 다음에 바꿔 볼 한 가지: {다음_질문 or "미입력"}
'''
note_path = OUTPUT_DIR / "lesson08_research_note.md"
note_path.write_text(note + "\n", encoding="utf-8")
try:
    from IPython.display import Markdown, display
    display(Markdown(note))
except ImportError:
    print(note)
print("실험 기록 저장:", note_path)

complete_lab_step(6, 6, '`lesson08_research_note.md` 파일과 기록 내용이 표시되어야 합니다.')
